# Phase 2c. Sample-level Immune Ecosystem Analysis

이 노트북의 목적은 C1QC-associated TAM이 우세한 샘플과 SPP1-associated TAM이 우세한 샘플은 T cell,Treg, DC 등 다른 면역세포 구성에서 체계적인 차이를 보이는지를 확인하는 것이다.

본 분석은 필요한 이유는 다음과 같다.

Phase 2a~2b는 TAM subtype 자체의 annotation과 pan-cancer 분포를 보였다.  
Phase 2c는 TAM state 차이가 샘플 수준의 immune ecosystem과 어떻게 연결되는가로 질문을 한 단계 올린다.  
이 분석이 있어야 Phase 3의 spatial co-localization 질문으로 자연스럽게 이어질 수 있다.  

## 분석 한계 (사전 명시)

- GSE127465 유효 샘플: 전체 26개 중 Unresolved 100% 샘플(8개) 제외 -> 최대 n=18  
- n = 18은 통계적 파워가 제한적이므로 결과는 탐색적 분석으로 해석한다.  
- Phase 1 cell type annotation은 대분류(T cell / B cell / Macrophage / Cancer cell) 기준이므로 Treg / NK / DC 등 세부 구분은 현재 annotation 수준에서 직접 추출 불가 -> 가능한 대분류 내 비율 비교로 진행하고 세부 분류를 별도 subclustering이 필요함을 명시한다.  

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from utils.paths import *
from utils.report import ph2_report_adata_state

sc.settings.set_figure_params(dpi=100, facecolor='white')
np.random.seed(42)

# 1. 데이터 로드 및 상태 확인

두 가지 데이터를 사용한다.

- `HUMAN_H5AD`: Phase 1 전체 cell type annotation (T cell / B cell / Macrophage / Cancer cell / Unknown)
- `MAC_TME_H5AD`: Phase 2a macrophage subset + `tam_subtype_final` annotation

In [3]:
# 전체 cell type annotation 데이터
adata = sc.read_h5ad(HUMAN_H5AD)
ph2_report_adata_state(adata, 'Phase 1 full data')
print(adata.obs['cell_type'].value_counts())
print('sample count:', adata.obs['sample'].nunique())

===== Phase 1 full data =====
shape: 44,860 cells x 2,000 genes
X type: <class 'numpy.ndarray'>
X dtype: float64
obs names unique: True
var names unique: True
obs columns: ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_counts', 'sample', 'leiden', 'cell_type']
obsm keys: ['X_pca', 'X_pca_harmony', 'X_umap']
obsp keys: ['connectivities', 'distances']
layers keys: ['counts']
uns keys: ['cell_type_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap']

cell_type
Unknown        14937
T cell         12705
Macrophage      9567
B cell          4572
Cancer cell     3079
Name: count, dtype: int64
sample count: 26


In [4]:
# TAM subtype annotation 데이터
mac = sc.read_h5ad(MAC_TME_H5AD)
ph2_report_adata_state(mac, 'Phase 2a macrophage subset')

# tam_subtype_final 컬럼 확인
if 'tam_subtype_final' in mac.obs.columns:
    print(mac.obs['tam_subtype_final'].value_counts())
else:
    print('WARNING: tam_subtype_final 없음 — tam_subtype_merged 확인')
    print(mac.obs.columns.tolist())

===== Phase 2a macrophage subset =====
shape: 9,567 cells x 32,634 genes
X type: <class 'scipy.sparse._csr.csr_matrix'>
X dtype: float32
obs names unique: True
var names unique: True
obs columns: ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_counts', 'sample', 'leiden', 'cell_type', 'leiden_7', 'leiden_8', 'leiden_9', 'leiden_10', 'leiden_12', 'leiden_15', 'leiden_r0.1', 'leiden_r0.2', 'leiden_r0.3', 'leiden_r0.4', 'leiden_r0.5', 'leiden_r0.6', 'leiden_r0.7', 'leiden_r0.8', 'leiden_r0.9', 'leiden_r1.0', 'C1QC_score', 'SPP1_score', 'Resting_C1QC_score', 'Activated_C1QC_score', 'ISG15_score', 'tam_subtype', 'tam_subtype_initial', 'tam_subtype_final', 'tam_subtype_merged']
obsm keys: ['X_pca', 'X_pca_harmony', 'X_umap']
obsp keys: ['co

# 2. Sample-level TAM dominate type 결정

sample별 TAM subtype 비율을 계산하고 domiant type을 분류해보겠다.

유효 TAM subtype만을 사용하여 sample 별 TAM subtype을 센다. (tentative merge 된 데이터 이용)
즉 Unresolved는 이 단계에서 제외한다.

In [6]:
mac_valid = mac[mac.obs['tam_subtype_merged'] != 'Unresolved'].copy()

tam_counts = (
    mac_valid.obs
    .groupby(['sample', 'tam_subtype_merged'])
    .size()
    .unstack(fill_value=0)
)
tam_props = tam_counts.div(tam_counts.sum(axis=1), axis=0)

print(f"유효 샘플 수 (Unresolved 제외): {len(tam_props)}")
print(tam_props.round(3))

유효 샘플 수 (Unresolved 제외): 18
tam_subtype_merged  C1QC-associated TAM  SPP1+ TAM  \
sample                                               
0                                 0.522      0.237   
1                                 0.539      0.231   
2                                 0.428      0.269   
3                                 0.443      0.277   
7                                 0.636      0.364   
8                                 0.692      0.308   
10                                0.701      0.160   
11                                0.693      0.102   
12                                0.683      0.204   
14                                0.584      0.198   
15                                0.615      0.154   
16                                0.598      0.197   
18                                0.672      0.198   
19                                0.593      0.207   
20                                0.637      0.149   
21                                0.548      0.213   


C:\Users\82108\AppData\Local\Temp\ipykernel_11924\2442246586.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['sample', 'tam_subtype_merged'])


result
- C1QC-associated TAM이 모든 sample에서 우세하다,
- SPP1+ TAM은 샘플들 간의 변동이 있는 편이다.
- 완전히 SPP1 dominant한 샘플은 없다.

In [7]:
# 샘플별 macrophage 수 확인
mac_per_sample = mac_valid.obs.groupby('sample', observed=True).size()
print("샘플별 macrophage 수:")
print(mac_per_sample.sort_values())
print(f"\n< 10 cells 샘플: {(mac_per_sample < 10).sum()}개")
print(f"median cells/sample: {mac_per_sample.median():.0f}")

샘플별 macrophage 수:
sample
7      11
8      13
14    101
15    117
16    132
10    144
11    166
12    186
2     201
21    221
18    247
3     253
20    281
19    290
0     435
1     477
24    769
23    792
dtype: int64

< 10 cells 샘플: 0개
median cells/sample: 211


주의점
- 10개보다 적은 cell은 없어서 필터링 기준은 통과.
- 하지만 sample 7, 8은 결과 해석 시 주의 요함 -> cell 수가 적어서 비율 추정이 불안정하다.

# 3. 전체 adata에서 sample 별 cell type의 비율 계산

분석 대상인 sample들만 필터링하여 (아까 위에서 확인했듯 18개의 sample만 사용) sample별 cell type 비율을 확인한다.  
cell type 비율을 확인 한 후 분석 대상 cell type을 결정하는 순서로 진행한다. 
-> cell 자체가 너무 적을 경우 비율 추정이 불안정하여 해석에 방해가 될 수 있기 때문에.   

In [9]:
valid_samples = tam_props.index.tolist()
adata_valid = adata[adata.obs['sample'].isin(valid_samples)].copy()

print(f"전체 cells: {adata.n_obs:,} -> 유효 샘플 cells: {adata_valid.n_obs:,}")
print(f"샘플 수: {adata_valid.obs['sample'].nunique()}")

ct_counts = (
    adata_valid.obs
    .groupby(['sample', 'cell_type'], observed=True)
    .size()
    .unstack(fill_value=0)
)
ct_props = ct_counts.div(ct_counts.sum(axis=1), axis=0)

print("\n샘플별 cell type 비율:")
print(ct_props.round(3))
print("\ncell type 목록:", ct_props.columns.tolist())

전체 cells: 44,860 -> 유효 샘플 cells: 32,600
샘플 수: 18

샘플별 cell type 비율:
cell_type  B cell  Cancer cell  Macrophage  T cell  Unknown
sample                                                     
0           0.033        0.164       0.374   0.226    0.204
1           0.030        0.167       0.366   0.226    0.211
2           0.032        0.144       0.299   0.277    0.247
3           0.042        0.162       0.323   0.256    0.217
7           0.077        0.160       0.165   0.253    0.345
8           0.047        0.205       0.152   0.228    0.368
10          0.336        0.061       0.120   0.255    0.228
11          0.346        0.064       0.118   0.250    0.222
12          0.336        0.070       0.140   0.220    0.234
14          0.040        0.035       0.120   0.605    0.200
15          0.034        0.042       0.165   0.560    0.198
16          0.029        0.037       0.146   0.567    0.221
18          0.131        0.221       0.312   0.137    0.199
19          0.182        0.134  

cell type 비율 확인 후 분석 대상 결정:  
- B cell, Cancer cell, T cell, Macrophage는 분석에 포함  
- Unknown은 제외 -> annotation이 불명확하기 때문  
- Macrophage는 TAM을 포함하므로 C1QC 비율과 양의 상관이 나올 수 밖에 없기 때문에 포함하되 해석 시 명시적으로 주의요함  

# 4. 분석 매트릭스 구성

분석 대상으로 선정 된 cell type들을 따로 건져내고 C1QC비율과 cell type의 비율을 합친다.  
샘플 별 macrophage의 수를 추가한다. -> low-n 마킹용으로 사용  
cell이 30보다 적은 sample7과 8은 각각 low num으로 마킹한다.  

In [11]:
analysis_ct = ['B cell', 'Cancer cell', 'T cell', 'Macrophage']

df = pd.DataFrame({
    'C1QC_prop': tam_props['C1QC-associated TAM'],
    'SPP1_prop': tam_props['SPP1+ TAM'],
    'Mixed_prop': tam_props['C1QC/SPP1 mixed-signature TAM'],
})
df = df.join(ct_props[analysis_ct])

df['mac_n'] = mac_per_sample.reindex(df.index.astype(int)).values
df['low_n'] = df['mac_n'] < 30

df.index = df.index.astype(str)

print("분석 매트릭스:")
print(df.round(3))
print(f"\nlow-n 샘플 (mac < 30): {df['low_n'].sum()}개")

분석 매트릭스:
        C1QC_prop  SPP1_prop  Mixed_prop  B cell  Cancer cell  T cell  \
sample                                                                  
0           0.522      0.237       0.241   0.033        0.164   0.226   
1           0.539      0.231       0.231   0.030        0.167   0.226   
2           0.428      0.269       0.303   0.032        0.144   0.277   
3           0.443      0.277       0.281   0.042        0.162   0.256   
7           0.636      0.364       0.000   0.077        0.160   0.253   
8           0.692      0.308       0.000   0.047        0.205   0.228   
10          0.701      0.160       0.139   0.336        0.061   0.255   
11          0.693      0.102       0.205   0.346        0.064   0.250   
12          0.683      0.204       0.113   0.336        0.070   0.220   
14          0.584      0.198       0.218   0.040        0.035   0.605   
15          0.615      0.154       0.231   0.034        0.042   0.560   
16          0.598      0.197       0.205  